# Advanced Problems: `**kwargs`

This notebook contains advanced practice problems with full solutions.

Topics covered:

- Collecting arbitrary keyword arguments with `**kwargs`
- Combining `*args` and `**kwargs`
- Required keyword-only arguments before `**kwargs`
- Why `**kwargs` must come last
- Validating unknown keyword arguments
- Forwarding arguments to other functions
- Designing flexible but safe APIs
- Debugging `TypeError`s involving keyword arguments

## Problem 1: Predict the Output

Predict the output of the following calls before running the code.

In [1]:
def show_kwargs(**kwargs):
    print(kwargs)

show_kwargs(a=1, b=2)
show_kwargs(name='Python', version=3)
show_kwargs()

{'a': 1, 'b': 2}
{'name': 'Python', 'version': 3}
{}


### Solution 1

`**kwargs` collects keyword arguments into a dictionary.

If no keyword arguments are provided, `kwargs` is an empty dictionary.

In [2]:
# Expected output:
# {'a': 1, 'b': 2}
# {'name': 'Python', 'version': 3}
# {}

## Problem 2: Combine `*args` and `**kwargs`

Write a function `debug_call` that accepts any positional arguments and any keyword arguments.

It should print:

- The positional arguments as a tuple
- The keyword arguments as a dictionary

### Solution 2

In [3]:
def debug_call(*args, **kwargs):
    print('args:', args)
    print('kwargs:', kwargs)

debug_call(1, 2, 3, name='Alice', active=True)

args: (1, 2, 3)
kwargs: {'name': 'Alice', 'active': True}


`*args` collects extra positional arguments.

`**kwargs` collects extra keyword arguments.

`**kwargs` must appear after `*args` in the function definition.

## Problem 3: Explain the Invalid Signature

Why is the following function definition invalid?

In [4]:
# This is invalid Python syntax:
# def func(**kwargs, extra):
#     pass

### Solution 3

`**kwargs` must be the final parameter in a function signature.

Once Python reaches `**kwargs`, it captures all remaining keyword arguments. Therefore, no parameter can appear after it.

## Problem 4: Required Keyword-Only Argument Plus `**kwargs`

Write a function `create_user`.

Requirements:

- It should require a keyword-only argument called `username`.
- It should collect any additional keyword arguments into `metadata`.
- It should return a dictionary containing the username and metadata.

### Solution 4

In [5]:
def create_user(*, username, **metadata):
    return {
        'username': username,
        'metadata': metadata
    }

print(create_user(username='ada', role='admin', active=True))
print(create_user(username='grace'))

{'username': 'ada', 'metadata': {'role': 'admin', 'active': True}}
{'username': 'grace', 'metadata': {}}


`username` is required because it has no default value.

It is keyword-only because it appears after the bare `*`.

`**metadata` collects any additional keyword arguments.

## Problem 5: Validate Allowed Keyword Arguments

Write a function `connect` that accepts required positional arguments `host` and `port`, plus arbitrary keyword arguments.

Only these keyword arguments should be allowed:

- `timeout`
- `ssl`
- `retries`

If an unknown keyword argument is provided, raise `TypeError`.

### Solution 5

In [6]:
def connect(host, port, **options):
    allowed = {'timeout', 'ssl', 'retries'}
    unknown = set(options) - allowed

    if unknown:
        raise TypeError(f'Unexpected keyword argument(s): {unknown}')

    return {
        'host': host,
        'port': port,
        'options': options
    }

print(connect('localhost', 5432, timeout=10, ssl=True))

# Uncomment to test the error:
# connect('localhost', 5432, debug=True)

{'host': 'localhost', 'port': 5432, 'options': {'timeout': 10, 'ssl': True}}


This pattern is useful when you want flexibility but still want to reject invalid option names.

## Problem 6: Add Defaults to `**kwargs`

Write a function `render_button` that accepts arbitrary keyword arguments.

Supported options:

- `text`, default `'Submit'`
- `color`, default `'blue'`
- `disabled`, default `False`

Return a dictionary with the final values.

Reject unknown options.

### Solution 6

In [7]:
def render_button(**kwargs):
    defaults = {
        'text': 'Submit',
        'color': 'blue',
        'disabled': False
    }

    unknown = set(kwargs) - set(defaults)
    if unknown:
        raise TypeError(f'Unknown option(s): {unknown}')

    config = defaults | kwargs
    return config

print(render_button())
print(render_button(text='Save', color='green'))
print(render_button(disabled=True))

{'text': 'Submit', 'color': 'blue', 'disabled': False}
{'text': 'Save', 'color': 'green', 'disabled': False}
{'text': 'Submit', 'color': 'blue', 'disabled': True}


`defaults | kwargs` creates a new dictionary where user-provided keyword arguments override defaults.

## Problem 7: Forward Keyword Arguments

Write a wrapper function `safe_print` that forwards all positional and keyword arguments to Python's built-in `print` function.

However, if the caller does not provide `sep`, use `' | '` as the default separator.

### Solution 7

In [8]:
def safe_print(*args, **kwargs):
    kwargs.setdefault('sep', ' | ')
    print(*args, **kwargs)

safe_print('A', 'B', 'C')
safe_print('A', 'B', 'C', sep='-')
safe_print('Hello', end='!\n')

A | B | C
A-B-C
Hello!


`setdefault` only adds `sep` if the caller did not already provide it.

`print(*args, **kwargs)` forwards both positional and keyword arguments.

## Problem 8: Separate Known and Unknown Keyword Arguments

Write a function `build_card`.

Requirements:

- Required keyword-only argument: `title`
- Optional keyword-only argument: `subtitle=None`
- Collect all remaining keyword arguments into `styles`
- Return a dictionary with `title`, `subtitle`, and `styles`

### Solution 8

In [9]:
def build_card(*, title, subtitle=None, **styles):
    return {
        'title': title,
        'subtitle': subtitle,
        'styles': styles
    }

print(build_card(title='Dashboard'))
print(build_card(title='Dashboard', subtitle='Admin', width=300, shadow=True))

{'title': 'Dashboard', 'subtitle': None, 'styles': {}}
{'title': 'Dashboard', 'subtitle': 'Admin', 'styles': {'width': 300, 'shadow': True}}


Specific keyword-only parameters are matched first.

Any remaining keyword arguments are collected by `**styles`.

## Problem 9: Debug Duplicate Keyword Arguments

Explain why the following code fails.

In [10]:
def display(name, **kwargs):
    print(name)
    print(kwargs)

data = {'name': 'Alice', 'age': 30}

# This fails:
# display('Bob', **data)

### Solution 9

The call fails because `name` receives two values:

- `'Bob'` is passed positionally to `name`
- `data` also contains `'name': 'Alice'`

Python does not allow the same parameter to receive multiple values.

In [11]:
# Correct options:

display(**data)

clean_data = {'age': 30}
display('Bob', **clean_data)

Alice
{'age': 30}
Bob
{'age': 30}


When unpacking dictionaries with `**`, make sure the dictionary does not contain keys that conflict with explicitly supplied arguments.

## Problem 10: Implement a Strict Event Logger

Write a function `log_event`.

Requirements:

- Accept any number of positional message parts.
- Require keyword-only `level`.
- Collect extra keyword arguments into `context`.
- Valid levels are `'INFO'`, `'WARNING'`, and `'ERROR'`.
- Raise `ValueError` for invalid levels.
- Return a dictionary.

### Solution 10

In [12]:
def log_event(*message_parts, level, **context):
    valid_levels = {'INFO', 'WARNING', 'ERROR'}

    if level not in valid_levels:
        raise ValueError(f'Invalid level: {level}')

    return {
        'message': ' '.join(str(part) for part in message_parts),
        'level': level,
        'context': context
    }

print(log_event('User', 42, 'logged in', level='INFO', ip='127.0.0.1'))
print(log_event('Payment failed', level='ERROR', order_id='A100', retry=True))

{'message': 'User 42 logged in', 'level': 'INFO', 'context': {'ip': '127.0.0.1'}}
{'message': 'Payment failed', 'level': 'ERROR', 'context': {'order_id': 'A100', 'retry': True}}


This is a common real-world pattern:

- Use `*message_parts` for flexible message construction.
- Use required keyword-only arguments for important options.
- Use `**context` for extra structured metadata.

## Problem 11: Use `inspect.signature` with `**kwargs`

Use `inspect.signature` to inspect the kind of each parameter.

### Solution 11

In [13]:
import inspect

def example(a, b=10, *args, c, d=20, **kwargs):
    pass

sig = inspect.signature(example)

for name, parameter in sig.parameters.items():
    print(name, '->', parameter.kind, ', default =', parameter.default)

a -> POSITIONAL_OR_KEYWORD , default = <class 'inspect._empty'>
b -> POSITIONAL_OR_KEYWORD , default = 10
args -> VAR_POSITIONAL , default = <class 'inspect._empty'>
c -> KEYWORD_ONLY , default = <class 'inspect._empty'>
d -> KEYWORD_ONLY , default = 20
kwargs -> VAR_KEYWORD , default = <class 'inspect._empty'>


Expected parameter kinds:

- `a`: positional-or-keyword
- `b`: positional-or-keyword
- `args`: variadic positional
- `c`: keyword-only
- `d`: keyword-only
- `kwargs`: variadic keyword

## Problem 12: Final Challenge — Flexible API Request Builder

Write a function `api_request`.

Requirements:

- Required positional argument: `endpoint`
- Required keyword-only argument: `method`
- Optional keyword-only argument: `timeout=30`
- Collect any additional keyword arguments into `params`
- Valid methods are `'GET'`, `'POST'`, `'PUT'`, and `'DELETE'`
- Raise `ValueError` for invalid methods
- Raise `ValueError` if `timeout <= 0`
- Return a dictionary representing the request

### Solution 12

In [14]:
def api_request(endpoint, *, method, timeout=30, **params):
    valid_methods = {'GET', 'POST', 'PUT', 'DELETE'}

    method = method.upper()

    if method not in valid_methods:
        raise ValueError(f'Invalid method: {method}')

    if timeout <= 0:
        raise ValueError('timeout must be positive')

    return {
        'endpoint': endpoint,
        'method': method,
        'timeout': timeout,
        'params': params
    }

print(api_request('/users', method='GET', active=True, limit=10))
print(api_request('/users', method='POST', timeout=60, name='Alice', role='admin'))

{'endpoint': '/users', 'method': 'GET', 'timeout': 30, 'params': {'active': True, 'limit': 10}}
{'endpoint': '/users', 'method': 'POST', 'timeout': 60, 'params': {'name': 'Alice', 'role': 'admin'}}


This function combines several important ideas:

- `endpoint` is positional-or-keyword.
- `method` and `timeout` are keyword-only.
- `**params` captures arbitrary request parameters.
- Validation keeps the flexible API safe.

## Best Practices Summary

- Use `**kwargs` when a function needs to accept flexible keyword options.
- Put `**kwargs` last in the function signature.
- Combine explicit keyword-only parameters with `**kwargs` when some options are required or important.
- Validate unknown keyword arguments when only certain options are allowed.
- Use `*args, **kwargs` when writing wrappers or forwarding calls.
- Be careful with duplicate arguments when using dictionary unpacking with `**`.
- Prefer explicit named parameters when the accepted options are fixed and known.
- Use `**kwargs` intentionally; too much flexibility can make APIs harder to understand.